# Heuristic pricing evaluation plots

This notebook visualizes evaluation metrics from EISim runs where pricing is controlled by a heuristic policy.

It expects one directory per scenario/control setup, each containing episode folders with `Pricelogs_*` subfolders.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## Configure inputs

Set each label to the corresponding **evaluation output directory**.

Example paths:
- `../EISim/EISim_output/H_20results/output_H_20servers_evaluation`
- `../EISim/EISim_output/H_100results/output_H_100servers_evaluation`

In [ ]:
evaluation_sets = {
    # 'H-20 servers': Path('../EISim/EISim_output/H_20results/output_H_20servers_evaluation'),
    # 'H-100 servers': Path('../EISim/EISim_output/H_100results/output_H_100servers_evaluation'),
}

assert evaluation_sets, 'Add at least one evaluation directory to evaluation_sets.'

In [ ]:
def _agent_index(agent_name: str) -> int:
    digits = ''.join(ch for ch in agent_name if ch.isdigit())
    return int(digits) if digits else -1

def collect_episode_metrics(output_dir: Path) -> pd.DataFrame:
    episodes = sorted([p for p in output_dir.iterdir() if p.is_dir()])
    rows = []

    for episode_idx, episode_dir in enumerate(episodes, start=1):
        pricelog_dirs = sorted([p for p in episode_dir.iterdir() if p.is_dir() and 'Pricelogs' in p.name])

        for pricelog_dir in pricelog_dirs:
            scenario = '_'.join(pricelog_dir.name.split('_')[2:])
            agent_files = sorted(pricelog_dir.glob('*.csv'))

            cumulative_returns = []
            avg_prices = []

            for csv_file in sorted(agent_files, key=lambda p: _agent_index(p.name.split('_')[0])):
                df = pd.read_csv(csv_file)
                if {'CumulativeProfit', 'Price'}.issubset(df.columns):
                    cumulative_returns.append(float(df['CumulativeProfit'].iloc[-1]))
                    avg_prices.append(float(df['Price'].mean()))

            if cumulative_returns:
                rows.append({
                    'episode': episode_idx,
                    'scenario': scenario,
                    'total_cumulative_return': float(np.sum(cumulative_returns)),
                    'avg_agent_cumulative_return': float(np.mean(cumulative_returns)),
                    'avg_agent_price': float(np.mean(avg_prices)),
                    'n_agents': len(cumulative_returns),
                })

    if not rows:
        raise ValueError(f'No evaluation price logs found in: {output_dir}')

    return pd.DataFrame(rows)

def summarize_with_ci(values: np.ndarray, confidence_z: float = 1.96):
    values = np.asarray(values, dtype=float)
    mean = float(values.mean())
    if len(values) < 2:
        return mean, np.nan
    sem = float(values.std(ddof=1) / np.sqrt(len(values)))
    return mean, confidence_z * sem

In [ ]:
results = {}
for label, path in evaluation_sets.items():
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing path for {label}: {path}')
    results[label] = collect_episode_metrics(path)

display(pd.concat({k: v.head() for k, v in results.items()}, names=['set']).reset_index(level=0))

In [ ]:
summary_rows = []
for label, df in results.items():
    for scenario, sdf in df.groupby('scenario'):
        for metric in ['total_cumulative_return', 'avg_agent_cumulative_return', 'avg_agent_price']:
            mean, ci = summarize_with_ci(sdf[metric].values)
            summary_rows.append({
                'set': label,
                'scenario': scenario,
                'metric': metric,
                'mean': mean,
                'ci_95': ci,
                'episodes': len(sdf),
            })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
metric_titles = {
    'total_cumulative_return': 'Total cumulative return (platform)',
    'avg_agent_cumulative_return': 'Average cumulative return per agent',
    'avg_agent_price': 'Average price per agent',
}

for metric, title in metric_titles.items():
    mdf = summary_df[summary_df['metric'] == metric].copy()
    pivot_mean = mdf.pivot(index='set', columns='scenario', values='mean')
    pivot_ci = mdf.pivot(index='set', columns='scenario', values='ci_95')

    ax = pivot_mean.plot(kind='bar', figsize=(12, 5), yerr=pivot_ci, capsize=4)
    ax.set_title(f'Heuristic evaluation: {title}')
    ax.set_xlabel('Evaluation set')
    ax.set_ylabel(title)
    ax.legend(title='Scenario', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
for label, df in results.items():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True)
    for scenario, sdf in df.groupby('scenario'):
        sdf = sdf.sort_values('episode')
        axes[0].plot(sdf['episode'], sdf['total_cumulative_return'], label=scenario)
        axes[1].plot(sdf['episode'], sdf['avg_agent_cumulative_return'], label=scenario)
        axes[2].plot(sdf['episode'], sdf['avg_agent_price'], label=scenario)

    axes[0].set_title(f'{label}: total return by episode')
    axes[1].set_title(f'{label}: avg agent return by episode')
    axes[2].set_title(f'{label}: avg agent price by episode')
    for ax in axes:
        ax.set_xlabel('Episode')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('Value')
    axes[2].legend(title='Scenario', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()